In [10]:
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver
from langchain_core.tools import tool
from langchain_google_genai import ChatGoogleGenerativeAI
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from langchain_core.messages import SystemMessage
import logging
import os

In [9]:
import ast
import operator

_OPS = {
    ast.Add: operator.add,
    ast.Sub: operator.sub,
    ast.Mult: operator.mul,
    ast.Div: operator.truediv,
    ast.Pow: operator.pow,
    ast.Mod: operator.mod,
    ast.USub: operator.neg,
    ast.UAdd: operator.pos,
}

def safe_eval(expression: str) -> float:
    """Safely evaluate a basic arithmetic expression."""
    def _eval(node):
        if isinstance(node, ast.Constant) and isinstance(node.value, (int, float)):
            return node.value
        if isinstance(node, ast.BinOp) and type(node.op) in _OPS:
            return _OPS[type(node.op)](_eval(node.left), _eval(node.right))
        if isinstance(node, ast.UnaryOp) and type(node.op) in _OPS:
            return _OPS[type(node.op)](_eval(node.operand))
        raise ValueError(f"Unsupported expression: {expression}")

    tree = ast.parse(expression, mode="eval")
    return _eval(tree.body)


In [22]:

NOTES = [
    "students  names are Ram Shyam and gita",
    "LangGrwh so answers are grounded in your documents instead of guessed.",

    "Saarathi Academy runs a 12-week AI Engineering and Machine Learning course, two hours a day, "
    "in Old Baneshwor, Kathmandu.",

    "The safe way to run arithmetic from a model is a small ast-based evaluator, never Python's "
    "eval(), because a tool is a door into your system.",
]


In [8]:


# import sqlite3

# DB_PATH = "aug_27_notes.db"

# def init_db():
#     conn = sqlite3.connect(DB_PATH)
#     conn.execute("""
#         CREATE TABLE IF NOT EXISTS notes (
#             id INTEGER PRIMARY KEY AUTOINCREMENT,
#             text TEXT NOT NULL,
#             created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP
#         )
#     """)
#     conn.commit()
#     conn.close()

In [25]:


model = ChatGoogleGenerativeAI(
    model="gemini-3.5-flash",   # or "gemini-1.5-pro", "gemini-1.5-flash"
    google_api_key="",  
)



# from langchain_openrouter import ChatOpenRouter

# model = ChatOpenRouter(
#     model="stealth/ox-alpha",
#     api_key="sk-or-..."
# )

In [3]:
me = {"configurable": {"thread_id": "aug_27"}} 

In [26]:
_vec = TfidfVectorizer().fit(NOTES)          # build the index once
_M   = _vec.transform(NOTES)


@tool
def search_notes(query: str) -> str:
    """Search the user's course notes and return the most relevant note."""
    return NOTES[int(cosine_similarity(_vec.transform([query]), _M)[0].argmax())]

@tool
def calculator(expression: str) -> float:
    """Evaluate an arithmetic expression, e.g. '8 * 12'."""
    return safe_eval(expression)

@tool
def save_note(text: str) -> str:
    """Save a fact the user wants remembered for later."""
    NOTEBOOK.append(text); return "saved"



In [17]:
TOOLS = [search_notes, calculator, save_note]

In [44]:
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver

assistant = create_agent(model, tools=TOOLS,
                         checkpointer=InMemorySaver())   # tools + memory

me = {"configurable": {"thread_id": "aug_27th_test_class_v1"}}   # one conversation

# raw_response = assistant.invoke({"messages": [{"role": "user",
#                   "content": "what is my name"}]}, me)

# print("done")

In [38]:
last_message = raw_response['messages'][-1]

In [43]:
dict(last_message)['content'][0]['text']

"Based on your course notes, the listed student names are **Ram**, **Shyam**, and **Gita**. \n\nHowever, there isn't a specific note indicating which of these is yours, or if you go by another name! Let me know if you'd like me to save your name for future reference."

In [58]:
def execute_agent(query:str):
    
    message = {"messages": [{"role": "user",
                  "content": query}]}
        
    raw_response  = assistant.invoke(message, me)
    msg_extracted  = raw_response['messages'][-1]
    msg_extracted = dict(msg_extracted)['content'][0]['text']
    print(msg_extracted)
    print(" \n \n ====== Done executing agent ======")
    
    
    
    

In [60]:
import builtins

input = builtins.input


In [61]:
while True:
    try:
        input = (input("Enter question: "))
        execute_agent(input)
        del input
        
    except KeyboardInterrupt:
        print("\nProgram stopped safely by the user.")
        break

Enter question:  who are the students


Based on your notes, the students are **Ram**, **Shyam**, and **Gita**.
 
 
 ====== Done executing agent ======


Enter question:  what is rag?


Based on your notes, **RAG** stands for **Retrieval-Augmented Generation**. 

It is a process where the system **retrieves relevant text first**, and then lets the AI model answer using that specific text. This ensures that the answers are grounded directly in your documents and notes instead of being guessed.
 
 
 ====== Done executing agent ======


Enter question:  who is barak obama


Based on general knowledge (as there are no mentions of him in your notes), **Barack Obama** is an American politician who served as the **44th President of the United States** from 2009 to 2017. 

He is a member of the Democratic Party and made history as the first African-American president of the United States. Before his presidency, he served as a U.S. Senator representing Illinois and as an Illinois state senator.
 
 
 ====== Done executing agent ======


Enter question:  


/Users/kbshal/mero_space/sarathi_academy/.venv/lib/python3.10/site-packages/langchain_google_genai/chat_models.py:3274: UserWarning: HumanMessage with empty content was removed to prevent API error
  warnings.warn(


IndexError: list index out of range